Author: Alana Pooler
<br>
Purpose: Complete final project

# Final Project
(Add Overview)

In [8]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    SQLTransformer,
    Binarizer,
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    PCA
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

# initialize spark session
spark = SparkSession.builder.getOrCreate()

First we need to read in the 'power_ml_data' file using pandas and convert it to a spark data frame.

We will use the Power_Zone_3 variable as our response variable and all of the other variables as predictors.

In [9]:
# read in as pandas df
pdf = pd.read_csv("power_ml_data.csv")

# convert to spark df and view first few rows
df = spark.createDataFrame(pdf)
df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

Let's look at the data types of each column.

In [7]:
df.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



Before we can fit any models, we need to define some transformations, which we will put into a pipeline using MLlib. 

First, we need to rename our response variable to 'label'.

In [10]:
df = df.withColumnRenamed("Power_Zone_3", "label")

Next, we need to cast the Hour column as `DoubleType` since it is currently stored as `LongType`.

In [11]:
hour_cast = SQLTransformer(
    statement="""
    SELECT *, CAST(Hour AS DOUBLE) AS Hour_double
    FROM __THIS__
    """
)

Now we need to binarize the Hour column based on the column being less than 6.5 or not, which will essentially give us an indicator of night and day.

In [12]:
hour_bin = Binarizer(
    threshold = 6.5,
    inputCol="Hour_double",
    outputCol="Hour_binary"
)

We also want to one-hot encode the Month column. First we can use StringIndexer() to map the column values to numeric indices, and then we can use OneHotEncoder on that result.

In [13]:
month_indexer = StringIndexer(
    inputCol="Month",
    outputCol="Month_index"
)

month_encoder = OneHotEncoder(
    inputCols=["Month_index"],
    outputCols=["Month_ohe"]
)

Next we want to run a PCA fit on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows columns.

We will first use VectorAssembler() to put the variables into one column that we can use with the PCA() estimator.

In [14]:
# place variables to run PCA fit on in one column
pca_assembler = VectorAssembler(
    inputCols=[
        "Temperature",
        "Humidity",
        "Wind_Speed",
        "General_Diffuse_Flows",
        "Diffuse_Flows"
    ],
    outputCol="pca_features"
)

# run PCA
pca = PCA(
    k=2,
    inputCol="pca_features",
    outputCol="pca_output"
)

Now we can use VectorAssembler() to put all of our predictors into one 'features' column.

In [15]:
assembler = VectorAssembler(
    inputCols=[
        "pca_output",
        "Hour_binary",
        "Power_Zone_1",
        "Power_Zone_2",
        "Month_ohe"
    ],
    outputCol="features"
)

### Linear Regression Model

Now that we have defined all of the transformations, we can build our elastic net model using a pipeline to apply all of the transformations.

In [19]:
# define linear regression model
lr = LinearRegression(
    featuresCol="features",
    labelCol="label"
)

# build pipeline
pipeline = Pipeline(stages=[
    hour_cast,
    hour_bin,
    month_indexer,
    month_encoder,
    pca_assembler,
    pca,
    assembler,
    lr
])

Now we need to define the parameter grid and grid values. An elastic net linear model uses `regParam` and `elasticNetParam`, and we will test all combinations of the values 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1 for each parameter.

In [22]:
grid_values = [0, 0.05, 0.1, 0.25, 0.5,
               0.75, 0.9, 0.95, 0.98,
               0.99, 1]

param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, grid_values)
    .addGrid(lr.elasticNetParam, grid_values)
    .build()
)

Next we will fit the model using 5-fold cross validation with RMSE as the criterion. 

In [23]:
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=5
)

cv_model = cv.fit(df)

26/04/28 16:06:16 WARN CacheManager: Asked to cache already cached data.        
26/04/28 16:06:16 WARN CacheManager: Asked to cache already cached data.


Now we can look at the optimal parameter values for this model, as well as the RMSE from the best model.

In [31]:
best_model = cv_model.bestModel.stages[-1]

print("Best regParam:", best_model._java_obj.getRegParam())
print("Best elasticNetParam:", best_model._java_obj.getElasticNetParam())
print("Best CV RMSE:", min(cv_model.avgMetrics))

Best regParam: 0.05
Best elasticNetParam: 0.1
Best CV RMSE: 2148.221226408556


Next we will find the training set RMSE by using the fitted model as a transformer and evaluating on the entire training set.

In [32]:
preds_cv = cv_model.transform(df)
RegressionEvaluator().evaluate(preds_cv)

2147.097369189216

The values we are predicting are large (in the tens of thousands), so the RMSE is quite large as well. Let's look at the predictions, actual values, and the residuals (observed - predicted) to get a better idea of how close the model is getting to the actual values.

The predicted values aren't perfect, but they aren't too far off either. Most of the residuals are between 1000 and 2000 higher than the actual values.

In [39]:
results = preds_cv.withColumn(
    "residual",
    col("label") - col("prediction")
)

results.select(
    "label",
    "prediction",
    "residual"
).show(10)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386| 20878.17586735406|-637.2120073540609|
|20131.08434|18660.379942312455|1470.7043976875466|
|19668.43373|18204.938828115257|1463.4949018847437|
|18899.27711|17590.861480513926|1308.4156294860732|
|18442.40964|16997.558176806913|1444.8514631930884|
|18130.12048|16517.969783846114| 1612.150696153887|
|17945.06024|16093.550460700182|1851.5097792998167|
|17459.27711|15723.023132246006|1736.2539777539932|
|17025.54217|  15271.3962263227|1754.1459436773002|
|16794.21687|14938.723536853973| 1855.493333146027|
+-----------+------------------+------------------+
only showing top 10 rows
